## Transfer from Google Colab to Hugging Face

In [1]:
from google.colab import drive

drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
LORA_PATH  = "/content/gdrive/MyDrive/models/Infra"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.chat_template = (
    '{% for message in messages %}'
    '{% if loop.first and messages[0]["role"] == "system" %}'
    '{{ "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\\n\\n" + message["content"] + "<|eot_id|>" }}'
    '{% elif message["role"] == "user" %}'
    '{{ "<|start_header_id|>user<|end_header_id|>\\n\\n" + message["content"] + "<|eot_id|>" }}'
    '{% elif message["role"] == "assistant" %}'
    '{{ "<|start_header_id|>assistant<|end_header_id|>\\n\\n" + message["content"] + "<|eot_id|>" }}'
    '{% endif %}'
    '{% endfor %}'
    '{{ "<|start_header_id|>assistant<|end_header_id|>\\n\\n" }}'
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype      = torch.float16,  # fixed
    device_map = "auto"
)
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

messages = [
    {'role': 'system', 'content': 'You are an expert SQL engineer. Return only valid SQL.'},
    {'role': 'user',   'content': 'Schema: orders(order_id, customer_id, amount, created_at). Get total sales per customer for the last 30 days.'},
]

# Fix — get string first, then tokenize separately
prompt  = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs  = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = model.generate(
        input_ids      = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_new_tokens = 200,
        do_sample      = False,
        pad_token_id   = tokenizer.eos_token_id,
        eos_token_id   = tokenizer.eos_token_id,
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print("\n=== Generated SQL ===\n")
print(response.strip())

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== Generated SQL ===

```sql
SELECT 
  customer_id, 
  SUM(amount) as total_sales
FROM 
  orders
WHERE 
  created_at >= NOW() - INTERVAL 30 DAY
GROUP BY 
  customer_id;
```


In [3]:
model.save_pretrained("/content/sql-genie-lora")
tokenizer.save_pretrained("/content/sql-genie-lora")


('/content/sql-genie-lora/tokenizer_config.json',
 '/content/sql-genie-lora/chat_template.jinja',
 '/content/sql-genie-lora/tokenizer.json')

## Inference from Hugging Face

In [4]:
import torch
import textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import hf_hub_download

BASE_MODEL   = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_REPO = "pshashid/llama3.1B_SQL_Finetuned"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Load chat template from your HF repo
template_path = hf_hub_download(repo_id=ADAPTER_REPO, filename="chat_template.jinja")
with open(template_path, 'r') as f:
    tokenizer.chat_template = f.read()

print("Template loaded:")
print(repr(tokenizer.chat_template))  # verify \n\n not spaces

tokenizer.pad_token    = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype      = torch.float16,
    device_map = "auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()

def run_sql_inference(schema, query, max_new_tokens=200):
    messages = [
        {'role': 'system', 'content': f'You are an expert SQL engineer. Return only valid SQL. Schema context: {schema}'},
        {'role': 'user',   'content': query},
    ]
    prompt  = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs  = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


SQL_TEST_CASES = [
    {'schema': 'Table: orders(order_id, customer_id, amount, created_at)',
     'query' : 'Get total sales per customer for the last 30 days, ordered by highest spend.'},
    {'schema': 'Tables: employees(emp_id, name, dept_id, salary), departments(dept_id, dept_name)',
     'query' : 'Find average salary per department, only for departments with more than 5 employees.'},
    {'schema': 'Table: products(product_id, name, category, price, stock_quantity)',
     'query' : 'Which product categories have more than 100 total units in stock? Show count per category.'},
]

print('=' * 70)
for i, tc in enumerate(SQL_TEST_CASES, 1):
    print(f'\n[Test {i}]')
    print(f'  Schema : {tc["schema"]}')
    print(f'  Query  : {tc["query"]}')
    response = run_sql_inference(tc['schema'], tc['query'])
    print(f'  Output :\n{textwrap.indent(response, "    ")}')
    print('-' * 70)

Template loaded:
'{% for message in messages %}{% if loop.first and messages[0]["role"] == "system" %}{{ "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\\n\\n" + message["content"] + "<|eot_id|>" }}{% elif message["role"] == "user" %}{{ "<|start_header_id|>user<|end_header_id|>\\n\\n" + message["content"] + "<|eot_id|>" }}{% elif message["role"] == "assistant" %}{{ "<|start_header_id|>assistant<|end_header_id|>\\n\\n" + message["content"] + "<|eot_id|>" }}{% endif %}{% endfor %}{{ "<|start_header_id|>assistant<|end_header_id|>\\n\\n" }}'


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


[Test 1]
  Schema : Table: orders(order_id, customer_id, amount, created_at)
  Query  : Get total sales per customer for the last 30 days, ordered by highest spend.
  Output :
    ```sql
    SELECT customer_id, SUM(amount) as total_sales
    FROM orders
    WHERE created_at >= NOW() - INTERVAL 30 DAY
    GROUP BY customer_id
    ORDER BY total_sales DESC;
    ```

    This SQL query will return the total sales for each customer over the last 30 days, ordered by the highest spend. 

    Here's a breakdown of the query:

    - `created_at >= NOW() - INTERVAL 30 DAY`: This line filters the orders to only include those created within the last 30 days.
    - `GROUP BY customer_id`: This groups the results by customer ID, so we can calculate the total sales for each customer.
    - `SUM(amount) as total_sales`: This calculates the total sales for each customer.
    - `ORDER BY total_sales DESC`: This orders the results by the total sales in descending order, so the customers with the highes